# RNN 循环神经网络

来自视频 [徒手实现循环神经网络--RNN的代码详解](https://www.bilibili.com/video/BV1R4421Q7tQ) 。、

推荐先观看视频 [飞天侠客的 RNN 介绍视频](https://www.bilibili.com/video/BV1NCgVzoEG9) 。

![](./images/RNN%20循环神经网络.png)
![](./images/RNN%20循环神经网络的解释.png)
![](./images/RNN%20循环神经网络的解释2.png)

我们都知道 RNN 网络就是将一个时间刻内产生的输出作为隐藏状态重新给到了下一个时刻，以此循环来保留序列中的顺序信息。

看着上面画的图实际上在刚开始了解的时候我陷入了一个误区：上面黄色的“神经元”只有一个，难道是重复将输出的内容给到仅有一个“神经元”进行处理吗？这样是不是效果不好？

实际上，上面的黄色的圆圈表示的是一个隐藏层，并非单一的神经元，同样，这个黄色圆圈可以换成多个隐藏层的全连接或者其他什么东西。

另外，在飞天侠客的视频中，在介绍 RNN 输出时（输出下一个文字时），将最终要输出的内容又经历一个层级的处理，实际上唐一旦老师的视频中在后期的代码实现中也是这样的，上面的图片没有画出来。

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import string

char2indx = {s: i for i, s in enumerate(string.ascii_lowercase)}

import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
import matplotlib.pyplot as plt

torch.manual_seed(12046)

# https://huggingface.co/datasets/code-search-net/code_search_net/tree/main/data
datasets = load_dataset('json', data_files='./datasets/python/final/jsonl/train/*.jsonl.gz')
datasets = datasets['train'].filter(lambda x: 'apache/spark' in x['repo'])

In [2]:
class RNNCell(nn.Module):

    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.i2h = nn.Linear(input_size + hidden_size, hidden_size)  # 线性层，这里代码只定义了一层

    def forward(self, input, hidden=None):
        # input: (1, I)  I 是输入文本特征个数
        # hidden: (1, H)  H 是隐藏层的特征个数，也可说是隐藏状态的个数
        if hidden is None:
            hidden = self.init_hidden()

        combined = torch.concat((input, hidden), dim=-1)  # (1, I + H)
        hidden = F.relu(self.i2h(combined))   # (1,  H)
        return hidden

    def init_hidden(self):
        return torch.zeros((1, self.hidden_size), device='cuda')


![](images/RNNCell.png)

In [9]:
r_model = RNNCell(2, 3).to('cuda')
data = torch.randn(4, 1, 2, device='cuda')  # 有 4 个 token ，每个 token 1 个 1 个批次输入， 每个 token 2 个特征，当然这里实际上是模拟数据，只是为了模型中的输入定义服务的

hidden = None

for i in range(data.shape[0]):
    hidden = r_model(data[i], hidden)
    print(hidden)

tensor([[0.0000, 0.0000, 0.1674]], device='cuda:0', grad_fn=<ReluBackward0>)
tensor([[0.0000, 0.0000, 0.0652]], device='cuda:0', grad_fn=<ReluBackward0>)
tensor([[0.1610, 0.3545, 0.4305]], device='cuda:0', grad_fn=<ReluBackward0>)
tensor([[0.0000, 0.3773, 0.5489]], device='cuda:0', grad_fn=<ReluBackward0>)


In [10]:
class CharRNN(nn.Module):  # CharRNN 做的就是将隐藏层变为文本输出

    def __init__(self, vs):  # vs 文本编号个数
        super().__init__()

        self.emb = nn.Embedding(vs, 30)  # 30 个特征
        self.rnn = RNNCell(30, 50)  # 30 个输入特征，50 个隐藏特征
        self.lm = nn.Linear(50, vs) # 输出的 50 个隐藏特征转化为文本编号概率

    def forward(self, x, hidden=None):
        # x: (1)
        # hidden: (1, 50)
        embeddings = self.emb(x)  # (1, 30)
        hidden = self.rnn(embeddings, hidden)  # (1, 50)
        out = self.lm(hidden)  # (1, vs)
        return out, hidden

In [11]:
class CharTokenizer:

    def __init__(self, data, end_ind=0):
        # data: list[str]
        # 得到所有的字符
        chars = sorted(list(set(''.join(data))))
        #self.char2ind = {s: i + 2 for i, s in enumerate(chars)}
        self.char2ind = {s: i + 1 for i, s in enumerate(chars)}
        #self.char2ind['<|b|>'] = begin_ind
        self.char2ind['<|e|>'] = end_ind
        self.ind2char = {v: k for k, v in self.char2ind.items()}
        #self.begin_ind = begin_ind
        self.end_ind = end_ind

    def encode(self, x):
        # x: str
        return [self.char2ind[i] for i in x]

    def decode(self, x):
        # x: int or list[x]
        if isinstance(x, int):
            return self.ind2char[x]
        return [self.ind2char[i] for i in x]

In [13]:
tokenizer = CharTokenizer(datasets['original_string'])
test_str = 'def f(x):'
tokens = tokenizer.encode(test_str)
tokens

[70, 71, 72, 2, 72, 10, 90, 11, 28]

In [15]:
c_model = CharRNN(len(tokenizer.char2ind)).to('cuda')
c_model  # 第三层在大语言模型处理中叫做语言建模层（language modeling head）

CharRNN(
  (emb): Embedding(98, 30)
  (rnn): RNNCell(
    (i2h): Linear(in_features=80, out_features=50, bias=True)
  )
  (lm): Linear(in_features=50, out_features=98, bias=True)
)

In [16]:
inputs = torch.tensor(tokenizer.encode('d')).to('cuda')
out, hidden = c_model(inputs)
out.shape, hidden.shape

(torch.Size([1, 98]), torch.Size([1, 50]))

In [17]:
@torch.no_grad()
def generate(model, idx, tokenizer, max_new_tokens=300):
    # idx: (1)
    out = idx.tolist()
    hidden = None
    model.eval()
    for _ in range(max_new_tokens):
        logits, hidden = model(idx, hidden)
        probs = F.softmax(logits, dim=-1)  # (1, 98)
        # 随机生成文本
        ix = torch.multinomial(probs, num_samples=1)  # (1, 1)  给你一组概率，它随机选出一个或多个“最可能”的索引，选中的概率正比于这些值。
        ## 更新背景
        #context = torch.concat((context[:, 1:], ix), dim=-1)
        out.append(ix.item())
        idx = ix.squeeze(0)
        if out[-1] == tokenizer.end_ind:
            break
    model.train()
    return out

In [26]:
inputs = torch.tensor(tokenizer.encode('d'), device='cuda')
print(''.join(tokenizer.decode(generate(c_model, inputs, tokenizer))))

d]0f.
           b         seter the n|d ._ustrithe setib fam poamhat if ret py Sx :Ist = 2utung(mon cal. = arewhe the deyt and inionthe thin(sectise the alect:, Uinn joliond))
     Sp iolsemum =          `hetrrat it chams uor         efuinnevartea vextarnf in balionde))
           on (setoe =onCHape


In [23]:
def process(text, tokenizer):
    # text: str
    enc = tokenizer.encode(text)
    inputs = enc
    labels = enc[1:] + [tokenizer.end_ind]
    return torch.tensor(inputs, device='cuda'), torch.tensor(labels, device='cuda')

In [24]:
lossi = []
epochs = 1
optimizer = optim.Adam(c_model.parameters(), lr=0.001)
for e in range(epochs):
    for data in datasets:
        inputs, labels = process(data['original_string'], tokenizer)
        hidden = None
        _loss = 0.0
        lens = len(inputs)
        for i in range(lens):
            logits, hidden = c_model(inputs[i].unsqueeze(0), hidden)
            _loss += F.cross_entropy(logits, labels[i].unsqueeze(0)) / lens
        lossi.append(_loss.item())
        optimizer.zero_grad()
        _loss.backward()
        optimizer.step()
